In [8]:
import pandas as pd
from pathlib import Path
import os

# Show current working directory and candidate paths to help debug missing-file errors
print("Working directory:", os.getcwd())

candidates = [
    Path("data/synthetic_fraud_dataset_final.csv"),
    Path("../data/synthetic_fraud_dataset_final.csv"),
    Path("../../data/synthetic_fraud_dataset_final.csv"),
]
for p in candidates:
    print(p, "exists:", p.exists())

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "CSV not found. Make sure the notebook's working directory is the repository root or adjust the relative path.\n" 
        "Run `import os; print(os.getcwd())` and check the 'data' folder path relative to it."
    )

Working directory: c:\Users\mdani\data minor\DataMiningProject\notebooks\orai
data\synthetic_fraud_dataset_final.csv exists: False
..\data\synthetic_fraud_dataset_final.csv exists: False
..\..\data\synthetic_fraud_dataset_final.csv exists: True


In [9]:
print("Using:", csv_path)
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

Using: ..\..\data\synthetic_fraud_dataset_final.csv
(1000, 29)


,transaction_id,timestamp,customer_id,transaction_amount,account_balance_before,account_balance_after,transaction_frequency_24h,avg_transaction_amount_7d,num_failed_transactions_7d,time_since_last_transaction_hr,...,merchant_category,device_type,location_match,payment_channel,transaction_description,merchant_name,customer_support_note,is_fraud,risk_category,support_sentiment
0,TX000000,2025-01-01 08:39:39,CUST0160,83.82,3352.85,3269.03,0,0.0,0,2840.66,...,Food,mobile,1,card,Morning breakfast and coffee order for the off...,Corner Bakery Cafe,Standard morning purchase.,0,low,neutral
1,TX000001,2025-01-01 07:28:40,CUST0004,19.65,436.64,416.99,0,0.0,0,7351.48,...,Travel,mobile,1,card,Monthly transit pass renewal for commuter trai...,Metro Transit Authority,NaN,0,low,neutral
2,TX000002,2025-01-01 19:15:23,CUST0289,192.09,2401.12,2209.03,0,0.0,0,91.26,...,Clothing,desktop,1,card,Purchase of a new denim jacket and accessories.,Urban Outfitters,NaN,0,low,neutral
3,TX000003,2025-01-01 03:24:49,CUST0202,1612.91,1897.54,284.63,0,0.0,0,3819.41,...,Electronics,desktop,0,card,URGENT: High-end luxury watch purchase from ov...,Luxury Swiss Boutique,Transaction originated from a known high-risk ...,1,high,negative
4,TX000004,2025-01-01 10:12:35,CUST0092,331.14,2759.51,2428.37,0,0.0,0,2482.21,...,Utilities,desktop,1,bank,Payment for monthly residential electricity an...,City Power & Light,Recurring monthly utility payment.,0,low,neutral


In [13]:

X=df.drop(columns="support_sentiment")
Y=df["support_sentiment"]

In [18]:
# Ensure lists
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# Safe OneHotEncoder (handles sklearn versions)
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

# Build transformers only when columns exist
transformers = []
if num_cols:
    transformers.append(('num', StandardScaler(), num_cols))
if cat_cols:
    transformers.append(('cat', encoder, cat_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='passthrough')
arr = preprocessor.fit_transform(X)

# Create DataFrame only when names length matches columns
feature_names = []
if num_cols:
    feature_names += num_cols
if cat_cols:
    cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
    feature_names += cat_names

if len(feature_names) == arr.shape[1]:
    preprocessed_df = pd.DataFrame(arr, columns=feature_names)
else:
    preprocessed_df = pd.DataFrame(arr)

In [20]:
# Save preprocessed DataFrame to CSV at the requested absolute path and provide a download link
from pathlib import Path
from IPython.display import FileLink, display

out_path = Path(r"C:\Users\mdani\data minor\DataMiningProject\data\preprocessed_df.csv").resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)
preprocessed_df.to_csv(out_path, index=False)
print("Saved preprocessed_df to:", out_path)

display(FileLink(out_path))

Saved preprocessed_df to: C:\Users\mdani\data minor\DataMiningProject\data\preprocessed_df.csv


C:\Users\mdani\data minor\DataMiningProject\data\preprocessed_df.csv

In [21]:
preprocessed_df.head()

,transaction_amount,account_balance_before,account_balance_after,transaction_frequency_24h,avg_transaction_amount_7d,num_failed_transactions_7d,time_since_last_transaction_hr,transaction_hour,is_weekend,account_age_days,...,customer_support_note_User-initiated recurring investment.,customer_support_note_Verified KYC account.,customer_support_note_Verified KYC customer.,customer_support_note_Verified investment account.,customer_support_note_Verified loyalty card used.,customer_support_note_Verified utility provider.,customer_support_note_nan,risk_category_high,risk_category_low,risk_category_medium
0,-0.243940,0.844750,0.960212,-0.187608,-0.195533,-0.070888,0.491433,-0.773952,-0.71934,-0.920825,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.368252,-1.209796,-1.106698,-0.187608,-0.195533,-0.070888,2.367092,-0.945446,-0.71934,0.776132,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,-0.034196,0.174231,0.192016,-0.187608,-0.195533,-0.070888,-0.651803,1.112481,-0.71934,-1.958857,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,2.718258,-0.180554,-1.202621,-0.187608,-0.195533,-0.070888,0.898411,-1.631422,-0.71934,-0.550744,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.235176,0.426726,0.350975,-0.187608,-0.195533,-0.070888,0.342385,-0.430964,-0.71934,-1.056220,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
